# 10 - Utility Feature Selection

This notebook evaluates how clinical utility changes when we keep only the most important features for the utility task.


## Goal

Use the selected segmentation dataset and test smaller subsets of features ranked by utility importance.

Main questions:
- how many features are needed to preserve good utility?
- can we remove a substantial number of features without a strong clinical penalty?


In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / "src"))

import json
import pandas as pd

from config import FEATURE_SETS_DIR, FINAL_SEGMENT_FEATURES_DIR
from modeling import (
    compute_utility_feature_importance,
    get_feature_columns,
    run_utility_feature_subset_experiments,
)


In [4]:
FEATURES_DATASET_DIR = FINAL_SEGMENT_FEATURES_DIR
MAX_CHUNKS = None  # set an integer here for quick tests
SUBSET_SIZES = [10, 20, 30, 50, 75, 100, 150, 208]
UTILITY_MODELS = ["LogisticRegression", "XGBoost"]


## Load Selected Feature Dataset


In [5]:
manifest_path = FEATURES_DATASET_DIR / "manifest.json"
errors_path = FEATURES_DATASET_DIR / "errors.csv"

manifest = pd.read_json(manifest_path, typ="series")
chunk_files = [FEATURES_DATASET_DIR / chunk["chunk_file"] for chunk in manifest["chunks"]]
if MAX_CHUNKS is not None:
    chunk_files = chunk_files[:MAX_CHUNKS]

features_df = pd.concat(
    [pd.read_csv(chunk_file, compression="gzip", low_memory=False) for chunk_file in chunk_files],
    ignore_index=True,
)
errors_df = pd.read_csv(errors_path) if errors_path.exists() and errors_path.stat().st_size > 0 else pd.DataFrame()

training_features = get_feature_columns(features_df)

print("Window (s):", manifest.get("window_sec"))
print("Step (s):", manifest.get("step_sec"))
print("Chunk files loaded:", len(chunk_files))
print("Features dataframe:", features_df.shape)
print("Stored preprocessing errors:", len(errors_df))
print("Training features:", len(training_features))


Window (s): 2.0
Step (s): 1.0
Chunk files loaded: 181
Features dataframe: (406359, 215)
Stored preprocessing errors: 1
Training features: 208


## Rank Features By Utility Importance

We use the current utility baseline model to get an ordered list of features.


In [6]:
utility_importance = compute_utility_feature_importance(
    features_df=features_df,
    model_name="LogisticRegression",
    test_size=0.2,
    random_state=42,
    permutation_scoring="average_precision",
)

utility_importance["model_based_df"].head(20)


,feature,importance
0,global_max_energy,3.904029
1,lead_V5_std,3.897976
2,lead_V1_rms,3.748871
3,lead_V6_std,3.642603
4,lead_V5_rms,3.594814
5,global_std_energy,3.324987
6,global_std_mean,3.302672
7,global_std_std,3.242020
8,lead_V4_std,3.170446
9,lead_V6_rms,3.012295


In [7]:
ranked_features = utility_importance["model_based_df"]["feature"].tolist()
ranked_features[:20]


['global_max_energy',
 'lead_V5_std',
 'lead_V1_rms',
 'lead_V6_std',
 'lead_V5_rms',
 'global_std_energy',
 'global_std_mean',
 'global_std_std',
 'lead_V4_std',
 'lead_V6_rms',
 'global_std_area',
 'lead_III_rms',
 'global_max_area',
 'global_max_mean',
 'lead_V4_rms',
 'lead_aVL_rms',
 'global_mean_rms',
 'global_min_rms',
 'global_mean_std',
 'lead_III_std']

## Utility Performance With Top-k Features


In [8]:
subset_results = []

for model_name in UTILITY_MODELS:
    result_df = run_utility_feature_subset_experiments(
        features_df=features_df,
        ranked_features=ranked_features,
        subset_sizes=SUBSET_SIZES,
        model_name=model_name,
        test_size=0.2,
        random_state=42,
    ).copy()
    result_df.insert(0, "ranking_model", "LogisticRegression")
    subset_results.append(result_df)

utility_subset_results_df = pd.concat(subset_results, ignore_index=True)
utility_subset_results_df.sort_values(["model", "subset_size"]).reset_index(drop=True)


,ranking_model,model,subset_size,f1_score,f1_macro,balanced_accuracy,roc_auc,pr_auc,feature_columns
0,LogisticRegression,LogisticRegression,10,0.358614,0.537019,0.569695,0.602199,0.286035,"[global_max_energy, lead_V5_std, lead_V1_rms, ..."
1,LogisticRegression,LogisticRegression,20,0.373723,0.538502,0.580740,0.616216,0.293602,"[global_max_energy, lead_V5_std, lead_V1_rms, ..."
2,LogisticRegression,LogisticRegression,30,0.381671,0.542876,0.587887,0.627807,0.308106,"[global_max_energy, lead_V5_std, lead_V1_rms, ..."
3,LogisticRegression,LogisticRegression,50,0.545827,0.678236,0.734271,0.810247,0.545013,"[global_max_energy, lead_V5_std, lead_V1_rms, ..."
4,LogisticRegression,LogisticRegression,75,0.625269,0.740641,0.793587,0.873484,0.652413,"[global_max_energy, lead_V5_std, lead_V1_rms, ..."
5,LogisticRegression,LogisticRegression,100,0.658877,0.765480,0.817777,0.896568,0.696704,"[global_max_energy, lead_V5_std, lead_V1_rms, ..."
6,LogisticRegression,LogisticRegression,150,0.678075,0.779402,0.831441,0.906544,0.710191,"[global_max_energy, lead_V5_std, lead_V1_rms, ..."
7,LogisticRegression,LogisticRegression,208,0.678864,0.780043,0.831829,0.907604,0.712167,"[global_max_energy, lead_V5_std, lead_V1_rms, ..."
8,LogisticRegression,XGBoost,10,0.031821,0.454777,0.506417,0.641087,0.333078,"[global_max_energy, lead_V5_std, lead_V1_rms, ..."
9,LogisticRegression,XGBoost,20,0.057815,0.468000,0.512348,0.653330,0.350210,"[global_max_energy, lead_V5_std, lead_V1_rms, ..."


In [9]:
best_by_model = utility_subset_results_df.sort_values(["model", "f1_score"], ascending=[True, False]).groupby("model").head(1)
best_by_model.reset_index(drop=True)


,ranking_model,model,subset_size,f1_score,f1_macro,balanced_accuracy,roc_auc,pr_auc,feature_columns
0,LogisticRegression,LogisticRegression,208,0.678864,0.780043,0.831829,0.907604,0.712167,"[global_max_energy, lead_V5_std, lead_V1_rms, ..."
1,LogisticRegression,XGBoost,208,0.674664,0.798343,0.774787,0.925978,0.781488,"[global_max_energy, lead_V5_std, lead_V1_rms, ..."


## Recommended Utility Feature Subset

Pick the smallest subset that keeps utility close to the full-feature baseline.


In [10]:
full_feature_count = len(training_features)
full_rows = utility_subset_results_df[utility_subset_results_df["subset_size"] == full_feature_count].copy()
full_rows = full_rows[["model", "f1_score", "balanced_accuracy", "roc_auc", "pr_auc"]].rename(columns={
    "f1_score": "full_f1_score",
    "balanced_accuracy": "full_balanced_accuracy",
    "roc_auc": "full_roc_auc",
    "pr_auc": "full_pr_auc",
})

comparison_df = utility_subset_results_df.merge(full_rows, on="model", how="left")
comparison_df["f1_drop_vs_full"] = comparison_df["full_f1_score"] - comparison_df["f1_score"]
comparison_df["ba_drop_vs_full"] = comparison_df["full_balanced_accuracy"] - comparison_df["balanced_accuracy"]

comparison_df.sort_values(["model", "subset_size"]).reset_index(drop=True)


,ranking_model,model,subset_size,f1_score,f1_macro,balanced_accuracy,roc_auc,pr_auc,feature_columns,full_f1_score,full_balanced_accuracy,full_roc_auc,full_pr_auc,f1_drop_vs_full,ba_drop_vs_full
0,LogisticRegression,LogisticRegression,10,0.358614,0.537019,0.569695,0.602199,0.286035,"[global_max_energy, lead_V5_std, lead_V1_rms, ...",0.678864,0.831829,0.907604,0.712167,0.320249,0.262134
1,LogisticRegression,LogisticRegression,20,0.373723,0.538502,0.580740,0.616216,0.293602,"[global_max_energy, lead_V5_std, lead_V1_rms, ...",0.678864,0.831829,0.907604,0.712167,0.305141,0.251089
2,LogisticRegression,LogisticRegression,30,0.381671,0.542876,0.587887,0.627807,0.308106,"[global_max_energy, lead_V5_std, lead_V1_rms, ...",0.678864,0.831829,0.907604,0.712167,0.297193,0.243943
3,LogisticRegression,LogisticRegression,50,0.545827,0.678236,0.734271,0.810247,0.545013,"[global_max_energy, lead_V5_std, lead_V1_rms, ...",0.678864,0.831829,0.907604,0.712167,0.133036,0.097558
4,LogisticRegression,LogisticRegression,75,0.625269,0.740641,0.793587,0.873484,0.652413,"[global_max_energy, lead_V5_std, lead_V1_rms, ...",0.678864,0.831829,0.907604,0.712167,0.053595,0.038242
5,LogisticRegression,LogisticRegression,100,0.658877,0.765480,0.817777,0.896568,0.696704,"[global_max_energy, lead_V5_std, lead_V1_rms, ...",0.678864,0.831829,0.907604,0.712167,0.019986,0.014053
6,LogisticRegression,LogisticRegression,150,0.678075,0.779402,0.831441,0.906544,0.710191,"[global_max_energy, lead_V5_std, lead_V1_rms, ...",0.678864,0.831829,0.907604,0.712167,0.000789,0.000389
7,LogisticRegression,LogisticRegression,208,0.678864,0.780043,0.831829,0.907604,0.712167,"[global_max_energy, lead_V5_std, lead_V1_rms, ...",0.678864,0.831829,0.907604,0.712167,0.000000,0.000000
8,LogisticRegression,XGBoost,10,0.031821,0.454777,0.506417,0.641087,0.333078,"[global_max_energy, lead_V5_std, lead_V1_rms, ...",0.674664,0.774787,0.925978,0.781488,0.642843,0.268370
9,LogisticRegression,XGBoost,20,0.057815,0.468000,0.512348,0.653330,0.350210,"[global_max_energy, lead_V5_std, lead_V1_rms, ...",0.674664,0.774787,0.925978,0.781488,0.616848,0.262439


## Export Selected Utility Feature Set

We keep the `top-150` utility features as the main clinically oriented subset for the next stage.


In [11]:
SELECTED_SUBSET_SIZE = 150
FEATURE_SETS_DIR.mkdir(parents=True, exist_ok=True)

selected_utility_features = ranked_features[:SELECTED_SUBSET_SIZE]
selected_utility_feature_set_df = features_df[
    ["patient_id", "segment_id", "label", "utility_label", "segment_ref", "start_sample", "end_sample"]
    + selected_utility_features
].copy()

selected_features_json_path = FEATURE_SETS_DIR / "utility_top150_features.json"
selected_features_csv_path = FEATURE_SETS_DIR / "utility_top150_features.csv"
selected_dataset_csv_path = FEATURE_SETS_DIR / "utility_top150_dataset.csv.gz"
selected_summary_csv_path = FEATURE_SETS_DIR / "utility_feature_selection_summary.csv"

selected_features_json_path.write_text(
    json.dumps(
        {
            "subset_name": "utility_top150",
            "n_features": len(selected_utility_features),
            "ranking_model": "LogisticRegression",
            "window_sec": manifest.get("window_sec"),
            "step_sec": manifest.get("step_sec"),
            "features": selected_utility_features,
        },
        indent=2,
    ),
    encoding="utf-8",
)

pd.DataFrame({"feature": selected_utility_features}).to_csv(selected_features_csv_path, index=False)
selected_utility_feature_set_df.to_csv(selected_dataset_csv_path, index=False, compression="gzip")
utility_subset_results_df.to_csv(selected_summary_csv_path, index=False)

print("Saved feature list to:", selected_features_json_path)
print("Saved feature CSV to:", selected_features_csv_path)
print("Saved selected dataset to:", selected_dataset_csv_path)
print("Saved summary table to:", selected_summary_csv_path)
print("Selected dataset shape:", selected_utility_feature_set_df.shape)


Saved feature list to: C:\Users\Tiago\Documents\GitHub\ecg-privacy\data\processed\feature_sets\utility_top150_features.json
Saved feature CSV to: C:\Users\Tiago\Documents\GitHub\ecg-privacy\data\processed\feature_sets\utility_top150_features.csv
Saved selected dataset to: C:\Users\Tiago\Documents\GitHub\ecg-privacy\data\processed\feature_sets\utility_top150_dataset.csv.gz
Saved summary table to: C:\Users\Tiago\Documents\GitHub\ecg-privacy\data\processed\feature_sets\utility_feature_selection_summary.csv
Selected dataset shape: (406359, 157)


In [12]:
pd.DataFrame({"feature": selected_utility_features}).head(20)


,feature
0,global_max_energy
1,lead_V5_std
2,lead_V1_rms
3,lead_V6_std
4,lead_V5_rms
5,global_std_energy
6,global_std_mean
7,global_std_std
8,lead_V4_std
9,lead_V6_rms


## Notes

Suggested reading of the results:
- if a small top-k subset keeps almost the same `F1` and `balanced_accuracy`, the full feature set is likely redundant for utility;
- this subset can then become the starting point for the privacy-oriented feature removal stage.
